## Navigation

- **Section A**: Configuration and Setup
- **Section B**: BupaR Post-Target Analysis (Leakage Detection)
- **Section C**: Interactive Code Review and Filtering
- **Section D**: Generate Final Refined Feature Importances

## A. Configuration and Setup

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, HTML
import warnings
warnings.filterwarnings('ignore')

# Set project root
PROJECT_ROOT = Path('/home/pgx3874/pgx-analysis')  # Update for your EC2 path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Jupyter environment Python path (update if different)
PYTHON_BIN = Path('/home/pgx3874/jupyter-env/bin/python3.11')  # Full path to Jupyter Python
if not PYTHON_BIN.exists():
    # Fallback to sys.executable if custom path doesn't exist
    PYTHON_BIN = Path(sys.executable)
    print(f"⚠️  Custom Python path not found, using: {PYTHON_BIN}")

# Rscript path for BupaR analysis (update if different)
# The Python script will try to find Rscript automatically, but you can specify it here
# Based on EC2 setup: R is at /usr/local/bin/R, so Rscript is likely at /usr/local/bin/Rscript
RSCRIPT_BIN = Path('/usr/local/bin/Rscript')  # EC2 default location
if not RSCRIPT_BIN.exists():
    # Fallback to None for auto-detection if path doesn't exist
    RSCRIPT_BIN = None
    print(f"⚠️  Rscript not found at {Path('/usr/local/bin/Rscript')}, will use auto-detection")

# Import project utilities
from py_helpers.constants import age_band_to_fname

# Configuration — this notebook runs a single cohort only (opioid_ed 55-64)
COHORT = "opioid_ed"
AGE_BAND = "55-64"
AGE_BAND_FNAME = age_band_to_fname(AGE_BAND)

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "3b_feature_importance_eda" / "outputs" / COHORT / AGE_BAND_FNAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Configuration loaded")
print(f"   Cohort: {COHORT}")
print(f"   Age Band: {AGE_BAND} ({AGE_BAND_FNAME})")
print(f"   Python Binary: {PYTHON_BIN}")
if RSCRIPT_BIN:
    print(f"   Rscript Binary: {RSCRIPT_BIN}")
else:
    print(f"   Rscript Binary: Auto-detect (will be found by script)")
print(f"   Output Directory: {OUTPUT_DIR}")

### 1. Load Aggregated Feature Importances from Step 3

In [ ]:
# Load aggregated feature importance from Step 3
possible_paths = [
    PROJECT_ROOT / "3a_feature_importance" / "outputs" / COHORT / AGE_BAND / f"{COHORT}_{AGE_BAND_FNAME}_aggregated_feature_importance.csv",
    PROJECT_ROOT / "3a_feature_importance" / "from_s3" / "by_cohort" / COHORT / AGE_BAND / f"{COHORT}_{AGE_BAND_FNAME}_aggregated_feature_importance.csv",
]

aggregated_fi = None
for path in possible_paths:
    if path.exists():
        aggregated_fi = pd.read_csv(path)
        print(f"✅ Loaded aggregated feature importance from: {path}")
        print(f"   Total features: {len(aggregated_fi):,}")
        break

if aggregated_fi is None:
    print(f"❌ Could not find aggregated feature importance file")
    print(f"   Checked paths:")
    for path in possible_paths:
        print(f"     - {path}")
else:
    # Display summary
    print(f"\n📊 Feature Importance Summary:")
    print(f"   Columns: {list(aggregated_fi.columns)}")
    print(f"\n   Top 10 features:")
    display(aggregated_fi.head(10))

## C. BupaR Post-Target Analysis (Leakage Detection)

BupaR analysis identifies features that appear primarily after the target event (potential leakage).

### 1. Verify Rscript is Available

**Note:** BupaR analysis uses R scripts, so Rscript must be installed and available in PATH. The Python script will automatically find Rscript, but you can verify it's available here.

In [ ]:
# Verify Rscript is available
from py_helpers.rscript_utils import find_rscript, print_rscript_version

# Use the utility function to find Rscript (checks configured path, PATH, and common locations)
rscript_path = find_rscript(RSCRIPT_BIN if RSCRIPT_BIN else None)

if rscript_path:
    print(f"✅ Rscript found: {rscript_path}")
    print_rscript_version(rscript_path)
else:
    print(f"⚠️  Rscript not found")
    print(f"   The Python script will try to find it automatically")
    print(f"   You can set RSCRIPT_BIN environment variable to specify the path")

### 2. Run BupaR Post-Target Analysis

**Note:** This Python script calls R scripts (`create_bupar_outputs_*.R`) using Rscript. The script automatically finds Rscript, so no manual configuration is needed.

In [ ]:
import subprocess
from datetime import datetime

print("🚀 Running BupaR Post-Target Analysis...")
print(f"Started at: {datetime.now()}")
print(f"Note: This will call R scripts using Rscript")

cmd = [
    str(PYTHON_BIN),
    str(PROJECT_ROOT / "3b_feature_importance_eda" / "1_bupaR" / "run_bupar_post_target_analysis.py"),
    "--cohort", COHORT,
    "--age-band", AGE_BAND
]

result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print(f"\n✅ BupaR analysis completed successfully")
else:
    print(f"\n❌ BupaR analysis failed with return code {result.returncode}")
    if "Rscript not found" in result.stderr or "Rscript not found" in result.stdout:
        print("\n💡 Tip: Make sure R is installed and Rscript is in your PATH")
        print("   Or set RSCRIPT_BIN in the configuration cell above")

### 3. Load and Review BupaR Results

In [ ]:
# Load BupaR results
bupar_results_path = OUTPUT_DIR / f"{COHORT}_{AGE_BAND_FNAME}_bupar_post_target_analysis.csv"

if bupar_results_path.exists():
    bupar_results = pd.read_csv(bupar_results_path)
    print(f"✅ Loaded BupaR results: {len(bupar_results)} features analyzed")
    
    # Show post-target leakage features
    post_target_leakage = bupar_results[bupar_results.get('is_post_target_leakage', pd.Series([0]*len(bupar_results))) == 1]
    
    print(f"\n📊 BupaR Analysis Summary:")
    print(f"   Total features analyzed: {len(bupar_results)}")
    print(f"   Post-target leakage features: {len(post_target_leakage)}")
    
    if len(post_target_leakage) > 0:
        print(f"\n   ⚠️  Post-target leakage features identified:")
        display(post_target_leakage[['feature', 'is_post_target_leakage']].head(20))
    else:
        print(f"\n   ✅ No post-target leakage features identified")
    
    # Display full results
    print(f"\n   Full BupaR results:")
    display(bupar_results.head(20))
else:
    print(f"❌ BupaR results not found: {bupar_results_path}")
    bupar_results = pd.DataFrame()

### 4. View BupaR Visualizations

In [ ]:
# Display BupaR visualizations (opioid_ed: 5 plots including pre/post F1120)
bupar_plots = [
    f"{COHORT}_{AGE_BAND_FNAME}_overall_activity_frequency.png",
    f"{COHORT}_{AGE_BAND_FNAME}_activity_milestones_gantt.png",
    f"{COHORT}_{AGE_BAND_FNAME}_activity_sequence_top.png",
    f"{COHORT}_{AGE_BAND_FNAME}_pre_f1120_activity_frequency.png",
    f"{COHORT}_{AGE_BAND_FNAME}_post_f1120_activity_frequency.png",
]

# Check if plots directory exists and list available plots
if PLOTS_DIR.exists():
    available_plots = list(PLOTS_DIR.glob("*.png"))
    print(f"📁 Plots directory: {PLOTS_DIR}")
    print(f"   Found {len(available_plots)} PNG files")
    if available_plots:
        print(f"   Available plots:")
        for p in sorted(available_plots):
            print(f"     - {p.name}")
else:
    print(f"⚠️  Plots directory does not exist: {PLOTS_DIR}")

print(f"\n🔍 Looking for BupaR plots...")
for plot_name in bupar_plots:
    plot_path = PLOTS_DIR / plot_name
    if plot_path.exists():
        print(f"✅ Displaying: {plot_name}")
        display(Image(str(plot_path)))
    else:
        print(f"⚠️  Plot not found: {plot_path}")
        # Try alternative path (in case plots are in features/ subdirectory)
        alt_path = OUTPUT_DIR / "features" / plot_name
        if alt_path.exists():
            print(f"   Found in alternative location: {alt_path}")
            display(Image(str(alt_path)))
        else:
            # Try without cohort prefix (in case R script saved without it)
            simple_name = plot_name.replace(f"{COHORT}_{AGE_BAND_FNAME}_", "")
            simple_path = PLOTS_DIR / simple_name
            if simple_path.exists():
                print(f"   Found with simple name: {simple_path}")
                display(Image(str(simple_path)))

## D. Interactive Code Review and Filtering

Review the analysis results and manually add/remove codes that should be filtered before Step 4a.

### 1. Review Codes to Filter

Based on the BupaR analyses, review codes that should be filtered:

### 2. Manually Add/Remove Codes to Filter

**Instructions:**
1. Review the visualizations and analysis results above
2. Add codes to filter in the cell below (one per line)
3. Remove codes from the filtering list if they should be kept
4. Run the cell to update the filtering list

In [ ]:
# ============================================
# MANUAL CODE FILTERING
# ============================================
# Add codes here that you want to filter based on your review
# Format: one code per line as a string
# 
# NOTE: This section is idempotent - it will load existing config if it exists
# and merge new codes with existing ones. Running multiple times is safe.

# Initialize filtering_recommendations if not already defined
# Works in both scripts and notebooks
try:
    # Check if it exists by trying to access it
    _ = filtering_recommendations
except NameError:
    # Doesn't exist, create it
    filtering_recommendations = {
        'administrative_codes': set(),
        'bupar_post_target': set(),
        'manual_additional': set()
    }
    print("ℹ️  Initialized filtering_recommendations with empty sets")


# Load existing config if it exists (idempotency)
filtering_config_path = OUTPUT_DIR / f"{COHORT}_{AGE_BAND_FNAME}_manual_filtering_config.json"
existing_manual_codes = []
existing_codes_to_keep = []

if filtering_config_path.exists():
    try:
        with open(filtering_config_path, 'r') as f:
            existing_config = json.load(f)
        existing_codes_to_keep = existing_config.get('codes_to_keep', [])
        existing_manual_codes = []
        # Extract manual codes from existing config
        if 'manual_additional_count' in existing_config:
            manual_count = existing_config['manual_additional_count']
            all_codes = existing_config.get('codes_to_filter', [])
            admin_count = existing_config.get('administrative_codes_count', 0)
            bupar_count = existing_config.get('bupar_post_target_count', 0)
            if len(all_codes) > (admin_count + bupar_count):
                start_idx = admin_count + bupar_count
                existing_manual_codes = all_codes[start_idx:]
        print(f"✅ Loaded existing config: {len(existing_manual_codes)} manual codes, {len(existing_codes_to_keep)} codes to keep")
    except Exception as e:
        print(f"⚠️  Could not load existing config: {e}")
        print(f"   Starting with empty lists")

# Default: empty lists (can be overridden by user)
MANUAL_CODES_TO_FILTER = [
    # Example: "Z00.00",  # Administrative code
    # Example: "V70.0",   # Routine exam
    # Add your codes here:
]

# Remove codes from filtering if they should be kept
CODES_TO_KEEP = [
    # Example: "F11.20",  # Keep this code even if flagged
    # Add codes to keep here:
]

# Merge with existing codes (idempotent - union operation)
# Start with existing manual codes, then add new ones
all_manual_codes = set(existing_manual_codes) | set(MANUAL_CODES_TO_FILTER)
all_codes_to_keep = set(existing_codes_to_keep) | set(CODES_TO_KEEP)

# Update filtering recommendations (merge, don't replace)
filtering_recommendations['manual_additional'].update(all_manual_codes)

# Remove codes that should be kept from all categories
for code in all_codes_to_keep:
    filtering_recommendations['administrative_codes'].discard(code)
    filtering_recommendations['bupar_post_target'].discard(code)
    filtering_recommendations['manual_additional'].discard(code)

# Final list of codes to filter (union of all categories)
final_codes_to_filter = (
    filtering_recommendations['administrative_codes'] |
    filtering_recommendations['bupar_post_target'] |
    filtering_recommendations['manual_additional']
)

print(f"✅ Updated filtering list")
print(f"   Total codes to filter: {len(final_codes_to_filter)}")
print(f"     - Administrative codes: {len(filtering_recommendations['administrative_codes'])}")
print(f"     - BupaR post-target codes: {len(filtering_recommendations['bupar_post_target'])}")
print(f"     - Manual additional codes: {len(filtering_recommendations['manual_additional'])}")
print(f"     - Codes to keep: {len(all_codes_to_keep)}")

if len(final_codes_to_filter) > 0:
    print(f"\n   Codes to filter (showing first 50):")
    for code in sorted(list(final_codes_to_filter))[:50]:
        print(f"     - {code}")
    if len(final_codes_to_filter) > 50:
        print(f"     ... and {len(final_codes_to_filter) - 50} more")

# Save filtering list to JSON for use in next step (idempotent - overwrites with complete state)
filtering_config = {
    'codes_to_filter': sorted(list(final_codes_to_filter)),
    'codes_to_keep': sorted(list(all_codes_to_keep)),
    'administrative_codes_count': len(filtering_recommendations['administrative_codes']),
    'bupar_post_target_count': len(filtering_recommendations['bupar_post_target']),
    'manual_additional_count': len(filtering_recommendations['manual_additional'])
}

with open(filtering_config_path, 'w') as f:
    json.dump(filtering_config, f, indent=2)

print(f"\n   💾 Saved filtering config to: {filtering_config_path}")



### 3. Update Filtering Scripts (if needed)

If you've added manual codes, you may need to update the filtering scripts to include them. Otherwise, proceed to run the filter and refine step.

### 4. Run Filter and Refine

In [ ]:
import subprocess
from datetime import datetime

print("🚀 Filtering and Refining Features...")
print(f"Started at: {datetime.now()}")

cmd = [
    str(PYTHON_BIN),
    str(PROJECT_ROOT / "3b_feature_importance_eda" / "2_filtering" / "filter_and_refine_features.py"),
    "--cohort", COHORT,
    "--age-band", AGE_BAND
]

result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

if result.returncode == 0:
    print(f"\n✅ Filter and refine completed successfully")
else:
    print(f"\n❌ Filter and refine failed with return code {result.returncode}")

In [ ]:
# Load final refined feature importance
refined_fi_path = OUTPUT_DIR / f"{COHORT}_{AGE_BAND_FNAME}_cohort_feature_importance.csv"

if refined_fi_path.exists():
    refined_fi = pd.read_csv(refined_fi_path)
    print(f"✅ Loaded refined feature importance: {len(refined_fi)} features")
    
    # Load filtering summary
    summary_path = OUTPUT_DIR / f"{COHORT}_{AGE_BAND_FNAME}_feature_filtering_summary.json"
    if summary_path.exists():
        with open(summary_path, 'r') as f:
            filtering_summary = json.load(f)
        
        print(f"\n📊 Filtering Summary:")
        print(f"   Original features: {filtering_summary.get('original_count', 'N/A')}")
        print(f"   Filtered by post-target: {filtering_summary.get('filtered_by_post_target', 0)}")
        print(f"   Filtered by non-value-added: {filtering_summary.get('filtered_by_non_value_added', 0)}")
        print(f"   Filtered by threshold: {filtering_summary.get('filtered_by_threshold', 0)}")
        print(f"   Final features: {filtering_summary.get('final_count', 'N/A')}")
    
    print(f"\n   Top 20 refined features:")
    display(refined_fi.head(20))
    
    print(f"\n   ✅ File ready for Step 4a: {refined_fi_path}")
else:
    print(f"❌ Refined feature importance not found: {refined_fi_path}")

### 5. Verify S3 Upload

Check that the refined feature importance file was uploaded to S3 for Step 4a consumption.

In [ ]:
import boto3

s3_client = boto3.client('s3')
s3_bucket = 'pgxdatalake'
s3_key = f"gold/feature_importance/{COHORT}/{AGE_BAND}/{COHORT}_{AGE_BAND_FNAME}_cohort_feature_importance.csv"

try:
    s3_client.head_object(Bucket=s3_bucket, Key=s3_key)
    print(f"✅ File exists in S3: s3://{s3_bucket}/{s3_key}")
    
    # Get file size
    response = s3_client.head_object(Bucket=s3_bucket, Key=s3_key)
    size_mb = response['ContentLength'] / (1024 * 1024)
    print(f"   File size: {size_mb:.2f} MB")
    print(f"   Last modified: {response['LastModified']}")
except s3_client.exceptions.ClientError as e:
    if e.response['Error']['Code'] == '404':
        print(f"❌ File not found in S3: s3://{s3_bucket}/{s3_key}")
    else:
        print(f"❌ Error checking S3: {e}")

## Summary

✅ **Step 3b Interactive Analysis Complete**

**Outputs Generated:**
- ✅ BupaR post-target analysis results  
- ✅ Refined `cohort_feature_importance.csv` for Step 4a
- ✅ Filtering summary JSON
- ✅ Visualizations (BupaR)

**Next Steps:**
- Proceed to **Step 4a: Model Data Creation** using the refined feature importances
- The `cohort_feature_importance.csv` file is available locally and in S3